# Сравнение моделей поверх `runs/`

Работает **без GPU и без данных соревнования** — только читает `notes.md`
(YAML frontmatter) из `runs/` и считает GFLOPs через `FlopCounterMode` на
случайных весах (`pretrained=False`, метаданные архитектуры, не качество).

Важное ограничение честности: `runs/` в этом репозитории хранит только
`notes.md` (метрики/чекпоинты —гигабайты, в `.gitignore`, см. `runs/` в
корневом `.gitignore` — «historical tracked notes remain tracked»). Поэтому
здесь **нет** сравнения per-image предсказаний между линией `disentangle_*` и
легаси-архитектурой `emcad_v1` — она обучалась в другой ветке
(`codex/emcad-baseline`) с другим кодом `src/`, и веса/предсказания локально не
хранятся. Сравнение ниже — это то, что реально проверяемо в этом репозитории:
исторические leaderboard-числа из карточек запусков + GFLOPs текущей
архитектуры, посчитанный живьём.


In [ ]:
# %% Зависимости — только CPU.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
               "pyyaml", "pandas", "matplotlib"], check=True)


In [ ]:
# %% Корень проекта.
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (ROOT / "src").is_dir():
    raise RuntimeError("Откройте ноутбук из корня репозитория или из notebooks/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)


## 1. Все leaderboard-числа, реально записанные в `runs/*/notes.md`

Читаем YAML frontmatter каждой карточки (`run`, `series`, `parent`, `change`,
`leaderboard_score`, `verdict`). Свободный текст карточек («Что проверяли» и
т.п.) в основном не заполнен (`ЗАПОЛНИТЬ`) — это не подделывается, показываем
только структурированные поля, которые реально несут значение.


In [ ]:
# %% Парсим frontmatter всех notes.md под runs/ (baseline + archive/*).
import re

import pandas as pd
import yaml


def read_frontmatter(path: Path) -> dict:
    text = path.read_text(encoding="utf-8")
    match = re.match(r"^---\n(.*?)\n---\n", text, re.S)
    if not match:
        return {}
    return yaml.safe_load(match.group(1)) or {}


records = []
for notes_path in sorted(ROOT.glob("runs/**/notes.md")):
    record = read_frontmatter(notes_path)
    record["path"] = str(notes_path.relative_to(ROOT))
    record["is_archive"] = "archive" in notes_path.parts
    records.append(record)

cards = pd.DataFrame(records)
cards = cards[["run", "series", "parent", "change", "leaderboard_score", "verdict", "is_archive", "path"]]
cards = cards.sort_values("leaderboard_score", ascending=False, na_position="last").reset_index(drop=True)
cards


## 2. Текущая архитектура (`jpeg640_v1`) vs legacy (`emcad_v1`)

`runs/baseline` (`pipeline_version: jpeg640_v1`, текущий `src/`, PVT-v2-B2 +
native JPEG + LocalFusion + EMCAD-декодер, `640×640`, 6 эпох) — прямой родитель
всей линии `disentangle_*`, включая финальный конфиг. Все записи из
`runs/archive/*` — более старая линия `emcad_v1`, обучавшаяся другим кодом
(`codex/emcad-baseline`). Сравнивать их числа напрямую как «одна модель лучше
другой» некорректно: разный код, и не факт, что тот же train/val split
(нынешний фиксированный протокол `validation_protocol_20260908` появился позже
части архивных запусков — см. `docs/validation_protocol.md`). Ниже — что видно
без подмены источников, с явной пометкой линии.


In [ ]:
# %% Явно помечаем линию (jpeg640_v1 vs emcad_v1) и печатаем таблицу по группам.
cards["line"] = cards["is_archive"].map({False: "jpeg640_v1 (текущая)", True: "emcad_v1 (архив, другой код)"})
summary = cards.groupby("line")["leaderboard_score"].agg(["count", "mean", "min", "max"])
summary


## 3. GFLOPs текущей архитектуры: baseline → DG-Force (arm D) → +cross-attention (arm M) → финал (760, r8)

Считаем `count_gflops` на случайно инициализированных весах (`pretrained=False`,
`torch.device('meta')` для скорости) — это метаданные архитектуры (число
операций), не качество. Каждая строка отличается от предыдущей ровно одним
архитектурным переключателем — то же самое подряд наследование конфигов, что и
в финальной цепочке (`configs/experiments/disentangle.md`).


In [ ]:
# %% Живой замер GFLOPs по цепочке архитектурных решений (без обучения).
import torch

from src.budget import count_gflops
from src.config import load_experiment_config
from src.training.builders import build_model

NATIVE_SIZE = (1080, 1920)

ARCH_STEPS = [
    ("baseline (640, без DG-Force)", "configs/baseline.yaml"),
    ("+ DG-Force fuse, arm D (640)", "configs/experiments/disentangle_fuse.yaml"),
    ("+ cross-attention 16→32, arm M (640)", "configs/experiments/disentangle_fuse_cross32.yaml"),
    ("+ encoder pvt_v2_b2_li, RGB 760 (arm M)", "configs/experiments/disentangle_b2_li760_long.yaml"),
    ("+ reduction=8 — финальная архитектура", "configs/experiments/disentangle_b2_li760_r8_long.yaml"),
]

rows = []
for label, config_path in ARCH_STEPS:
    cfg = load_experiment_config(ROOT / config_path)
    with torch.device("meta"):
        model = build_model(cfg.model, aux_weight=cfg.loss.aux_weight, pretrained=False).eval()
        gflops = count_gflops(model, cfg.dataset.image_size, native_size=NATIVE_SIZE)
    del model
    rows.append({"step": label, "config": config_path, "image_size": cfg.dataset.image_size, "gflops": gflops})

budget_table = pd.DataFrame(rows)
budget_table["delta_gflops"] = budget_table["gflops"].diff()
budget_table


Ожидаемая картина (см. также `configs/experiments/disentangle.md` и
`configs/experiments/disentangle_b2_li760_long.yaml`, комментарий про 760↔768):
arm D почти не меняет GFLOPs (модуль в режиме `fuse` добавляет только 1×1/depthwise
свёртки в узком горлышке), arm M добавляет cross-attention на одном страйде, а
основной скачок даёт увеличение `dataset.image_size` до 760 при переходе на
`pvt_v2_b2_li`. **Абсолютные числа таблицы выше пересчитаны именно на этой
машине/версии PyTorch — не переносите числа из старых markdown-карточек
(`disentangle.md`, `disentangle_b2_li760_long.yaml`) как факт без перепроверки**,
там указаны референсные замеры на другом железе/этапе кода.


## 4. Финальный результат — куда впишется после реального прогона

У финального run `disentangle_b2_li760_r8_all_data_hard_pixel_ft` нет
собственной валидации (`train_all_data=true`, см. `solution.ipynb`, раздел 7) —
поэтому его development AIC формально наследуется от стадии A
(`disentangle_b2_li760_r8_long`). Строка ниже — заготовка: впишите нужные числа
после того, как получите `runs/disentangle_b2_li760_r8_long/summary.json` и
реальный leaderboard-скор финального сабмита.


In [ ]:
# %% Заготовка итоговой строки — заполняется после реального прогона.
final_row = {
    "run": "disentangle_b2_li760_r8_all_data_hard_pixel_ft",
    "line": "jpeg640_v1 (текущая, финал)",
    "development_aic_stage_a": None,   # TODO: runs/disentangle_b2_li760_r8_long/summary.json -> best.aic
    "leaderboard_score": None,         # TODO: фактический скор после сабмита
    "gflops_native_1080x1920": None,   # TODO: взять из solution.ipynb, раздел 4 (та же архитектура)
}
final_row
